In [3]:
import os

# Point to the custom folder on your massive VSC data drive
lib_dir = os.path.join(os.environ.get('VSC_DATA'), 'thesis_libs')
os.makedirs(lib_dir, exist_ok=True)

# Install everything directly into the data drive to avoid quota limits
!pip install --no-cache-dir timm albumentations opencv-python-headless scikit-learn pandas tqdm --target={lib_dir}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 63.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 480.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 487.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 557.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 557.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 831.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 294.0 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 303.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import os

# Point to your VSC_DATA library folder
lib_dir = os.path.join(os.environ.get('VSC_DATA'), 'thesis_libs')

# Force install NumPy 1.x to fix the compatibility crash
!pip install --no-cache-dir "numpy<2" --target={lib_dir} --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 289.2 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [10]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "--target", "/data/leuven/375/vsc37504/thesis_libs",
                "--upgrade", "--force-reinstall",
                "numpy", "opencv-python-headless", "pandas", "scikit-learn",
                "--quiet"])
print("Reinstalled — restart the kernel now")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Reinstalled — restart the kernel now


In [5]:
import os
import sys

# 1. Point the main notebook to your VSC_DATA library folder
lib_dir = os.path.join(os.environ.get('VSC_DATA'), 'thesis_libs')
if lib_dir not in sys.path:
    sys.path.insert(0, lib_dir)

# 2. IMPORTANT: Inject this path into the environment variables!
# This ensures that PyTorch DataLoader (if num_workers > 0) can find albumentations & timm
os.environ['PYTHONPATH'] = f"{lib_dir}:{os.environ.get('PYTHONPATH', '')}"

# 3. Run your requested imports
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import gc
from tqdm.auto import tqdm

print("✓ All deep learning dependencies loaded successfully from VSC_DATA!")

ImportError: Failed to load PyTorch C extensions:
    It appears that PyTorch has loaded the `torch/_C` folder
    of the PyTorch repository rather than the C extensions which
    are expected in the `torch._C` namespace. This can occur when
    using the `install` workflow. e.g.
        $ python -m pip install --no-build-isolation -v . && python -c "import torch"

    This error can generally be solved using the `develop` workflow
        $ python -m pip install --no-build-isolation -v -e . && python -c "import torch"  # This should succeed
    or by running Python from a different directory.

In [6]:
import os

# --- 1. DATASET FOLDERS (VSC Absolute Paths) ---
BASE_DIR = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis"

CATHETER_MASK_DIR = os.path.join(BASE_DIR, "predicted_masks", "catheter", "train")
ATRIUM_MASK_DIR   = os.path.join(BASE_DIR, "predicted_masks", "atrium", "train")
LABEL_XLSX        = os.path.join(BASE_DIR, "atriumandcatheter.xlsx")

# Where to save the trained EfficientNet weights
SAVE_DIR = os.path.join(BASE_DIR, "Tip_Features_Output", "barche_efficientnet")
os.makedirs(SAVE_DIR, exist_ok=True)

# --- 2. HYPERPARAMETERS ---
IMG_SIZE = 600    # Barche's exact size
NUM_EPOCHS = 60   # Barche's exact setting
BATCH_SIZE = 8

In [9]:
import os

# Check if the folders from the previous cell actually exist
print("Catheter folder exists:", os.path.exists(CATHETER_MASK_DIR))
print("Atrium folder exists:  ", os.path.exists(ATRIUM_MASK_DIR))

# List what is actually in the parent 'predicted_masks' folder
parent = os.path.join(BASE_DIR, "predicted_masks")
print(f"\nContents of {parent}:")

if os.path.exists(parent):
    for item in os.listdir(parent):
        print(f"  {item}")
        subpath = os.path.join(parent, item)
        if os.path.isdir(subpath):
            # Safety check: don't print thousands of files and crash the notebook
            sub_files = os.listdir(subpath)
            print(f"    (Contains {len(sub_files)} items. Showing first 5...)")
            for sub in sub_files[:5]:
                print(f"    {sub}")
else:
    print("  parent folder not found — checking one level up (BASE_DIR)...")
    if os.path.exists(BASE_DIR):
        for item in os.listdir(BASE_DIR):
            print(f"  {item}")
    else:
        print("  BASE_DIR folder also not found. Check your absolute path!")

Catheter folder exists: True
Atrium folder exists:   True

Contents of /vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/predicted_masks:
  catheter
    (Contains 2 items. Showing first 5...)
    train
    test
  atrium
    (Contains 2 items. Showing first 5...)
    train
    test
  debug_visuals
    (Contains 1540 items. Showing first 5...)
    IMG-0023-00001_01_gray.png
    IMG-0023-00001_02_bin.png
    IMG-0023-00001_03_atrium.png
    IMG-0023-00001_04_centerline_overlay.png
    IMG-0023-00001_05_tip_mask.png
  efficientnet_tip_dot_v3
    (Contains 6 items. Showing first 5...)
    fold1_best.pt
    fold2_best.pt
    fold3_best.pt
    fold4_best.pt
    fold5_best.pt
  .ipynb_checkpoints
    (Contains 0 items. Showing first 5...)
  atriumandcatheter.xlsx
  catheter_tip_features_ml_ready.csv


In [10]:
labels_df = pd.read_excel(LABEL_XLSX)
# adjust column names to match your Excel file
# labels_df should have image_name and tip (1-ok, 0-no) columns
print(labels_df.head())
print(f"Total labels: {len(labels_df)}")

            ap_id  arch (1-ok, 0-no)  tip (1-ok, 0-no)
0  IMG-0001-00001                  1                 1
1  IMG-0003-00001                  1                 0
2  IMG-0005-00001                  1                 1
3  IMG-0007-00001                  0                 1
4  IMG-0009-00001                  0                 0
Total labels: 277


In [11]:
# Run this first to see the actual column names
print(labels_df.columns.tolist())
print(labels_df.head(3))

['ap_id', 'arch (1-ok, 0-no)', 'tip (1-ok, 0-no)']
            ap_id  arch (1-ok, 0-no)  tip (1-ok, 0-no)
0  IMG-0001-00001                  1                 1
1  IMG-0003-00001                  1                 0
2  IMG-0005-00001                  1                 1


In [12]:
import re

MASK_SIZE = (960, 960)
available_masks = set(os.listdir(CATHETER_MASK_DIR))

# Create a mapping from clean ap_id to the actual filename on disk
ap_id_to_filename = {}
for fname in available_masks:
    clean_id = re.sub(r"\.png$", "", fname)
    clean_id = re.sub(r"\s*-\s*PRIMARY$", "", clean_id)
    clean_id = re.sub(r"\.dcm$", "", clean_id).strip()
    ap_id_to_filename[clean_id] = fname

catheter_arrays = []
atrium_arrays   = []
labels_list     = []
ids_list        = []

skipped = 0
for _, row in labels_df.iterrows():
    ap_id = str(row["ap_id"]).strip()

    # Check if we have a real file for this ap_id
    if ap_id not in ap_id_to_filename:
        skipped += 1
        continue

    img_name = ap_id_to_filename[ap_id]
    label = int(row["tip (1-ok, 0-no)"])

    cath_mask   = cv2.imread(os.path.join(CATHETER_MASK_DIR, img_name), cv2.IMREAD_GRAYSCALE)
    atrium_mask = cv2.imread(os.path.join(ATRIUM_MASK_DIR, img_name), cv2.IMREAD_GRAYSCALE)

    if cath_mask is None or atrium_mask is None:
        print(f"WARNING: could not read {img_name}")
        skipped += 1
        continue

    if cath_mask.shape != MASK_SIZE:
        cath_mask = cv2.resize(cath_mask, MASK_SIZE, interpolation=cv2.INTER_NEAREST)
    if atrium_mask.shape != MASK_SIZE:
        atrium_mask = cv2.resize(atrium_mask, MASK_SIZE, interpolation=cv2.INTER_NEAREST)

    catheter_arrays.append(cath_mask)
    atrium_arrays.append(atrium_mask)
    labels_list.append(label)
    ids_list.append(img_name)

catheter_np = np.array(catheter_arrays)
atrium_np   = np.array(atrium_arrays)

print(f"Loaded  : {len(labels_list)} samples") # This will now output 220

Loaded  : 220 samples


In [13]:
class ClassificationDataset(Dataset):
    def __init__(self, catheter_predictions, atria_predictions,
                 labels, ids=None, transform=None):
        self.predictions  = catheter_predictions
        self.labels       = torch.tensor(labels, dtype=torch.float32)
        self.atrial_mask  = atria_predictions
        self.ids          = ids
        self.transform    = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        mask  = torch.from_numpy(self.predictions[idx]).float()
        atria = torch.from_numpy(self.atrial_mask[idx]).float()
        blank = torch.zeros_like(mask)
        feature = torch.stack([mask, atria, blank], dim=0)        # (3, H, W)
        feature_np = feature.permute(1, 2, 0).numpy()             # (H, W, 3)
        if self.transform:
            feature = self.transform(image=feature_np)['image']   # (3, H, W)
        if self.ids is not None:
            return feature, self.labels[idx], self.ids[idx]
        return feature, self.labels[idx]

In [14]:
def build_barche_efficientnet(fine_tune='last_two', device=None):
    """
    Replicates Barche's call_model() with fine_tune='last_two'
    Input  : 3-channel (catheter_mask, atrium_mask, zeros) 600x600
    Output : single logit
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = timm.create_model(
        "tf_efficientnet_b7",
        pretrained=True,
        in_chans=3,
        num_classes=1,
    )

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    if fine_tune == 'last_two':
        for name, param in model.named_parameters():
            if any(b in name for b in ["blocks.5","blocks.6",
                                        "conv_head","bn2","classifier"]):
                param.requires_grad = True
    elif fine_tune == 'head_only':
        for name, param in model.named_parameters():
            if any(b in name for b in ["conv_head","bn2","classifier"]):
                param.requires_grad = True

    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 1),
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Barche EfficientNet-B7: {trainable:,} / {total:,} trainable")
    return model.to(device)

In [15]:
import torch

# Define the hardware device (this will automatically detect your A100 GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: NVIDIA B200


In [16]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_NEAREST),
    A.HorizontalFlip(p=0.3),
    A.Rotate(limit=15, p=0.2, border_mode=cv2.BORDER_CONSTANT),
    A.ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_NEAREST),
    A.ToTensorV2()
])

labels_array  = np.array(labels_list)
skf           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results  = []
all_val_probs = np.zeros(len(labels_array))
all_val_preds = np.zeros(len(labels_array))

print(f"\nTraining Barche EfficientNet-B7 — full catheter mask")
print(f"IMG_SIZE={IMG_SIZE}, EPOCHS={NUM_EPOCHS}")

for fold, (train_idx, val_idx) in enumerate(
        skf.split(np.arange(len(labels_array)), labels_array), 1):

    print(f"\n── Fold {fold}/5 ────────────────────────────────")

    train_loader = DataLoader(
        ClassificationDataset(
            catheter_np[train_idx], atrium_np[train_idx],
            labels_array[train_idx].tolist(),
            ids=[ids_list[i] for i in train_idx],
            transform=train_transform
        ),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        ClassificationDataset(
            catheter_np[val_idx], atrium_np[val_idx],
            labels_array[val_idx].tolist(),
            ids=[ids_list[i] for i in val_idx],
            transform=val_transform
        ),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
    )

    model     = build_barche_efficientnet(fine_tune='last_two', device=device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=1e-4, weight_decay=1e-3)

    # Barche's exact LR schedule: warmup 5 epochs + cosine
    warmup   = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=5)
    cosine   = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - 5, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

    best_auc  = 0.0
    best_path = os.path.join(SAVE_DIR, f"fold{fold}_best.pt")

    for epoch in range(1, NUM_EPOCHS + 1):
        # Training
        model.train()
        for imgs, lbls, _ in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
            optimizer.zero_grad()
            nn.BCEWithLogitsLoss()(model(imgs), lbls).backward()
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        probs_list, true_list = [], []
        with torch.no_grad():
            for imgs, lbls, _ in val_loader:
                p = torch.sigmoid(model(imgs.to(device)).squeeze(1))
                probs_list.extend(p.cpu().numpy())
                true_list.extend(lbls.numpy())

        val_auc = roc_auc_score(true_list, probs_list)

        # SAVE best weights — this is what Barche's code was missing
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{NUM_EPOCHS}  "
                  f"val_auc={val_auc:.4f}  (best={best_auc:.4f})")

    # Load best and record OOF predictions
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    probs_list, true_list = [], []
    with torch.no_grad():
        for imgs, lbls, _ in val_loader:
            p = torch.sigmoid(model(imgs.to(device)).squeeze(1))
            probs_list.extend(p.cpu().numpy())
            true_list.extend(lbls.numpy())

    all_val_probs[val_idx] = probs_list
    all_val_preds[val_idx] = (np.array(probs_list) >= 0.5).astype(int)

    fold_results.append({
        "Fold": fold,
        "AUC":  round(roc_auc_score(true_list, probs_list), 4),
        "Acc":  round(accuracy_score(true_list,
                      (np.array(probs_list)>=0.5).astype(int)), 4),
    })
    print(f"  ✓ Fold {fold}: AUC={fold_results[-1]['AUC']:.4f}  "
          f"Acc={fold_results[-1]['Acc']:.4f}")

    del model, optimizer, scheduler
    torch.cuda.empty_cache()
    gc.collect()

# Print summary
print("\n=== Barche EfficientNet (full mask) — Results ===")
print(pd.DataFrame(fold_results).to_string(index=False))
oof_auc = roc_auc_score(labels_array, all_val_probs)
oof_acc = accuracy_score(labels_array, all_val_preds)
print(f"\nOOF AUC = {oof_auc:.4f}")
print(f"OOF Acc = {oof_acc:.4f}")

# Save OOF predictions for two-step classifier
np.save(os.path.join(SAVE_DIR, "oof_probs.npy"), all_val_probs)
np.save(os.path.join(SAVE_DIR, "oof_preds.npy"), all_val_preds)
print(f"\nWeights saved to: {SAVE_DIR}")
print(f"Files: fold1_best.pt through fold5_best.pt")


Training Barche EfficientNet-B7 — full catheter mask
IMG_SIZE=600, EPOCHS=60

── Fold 1/5 ────────────────────────────────
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
  Epoch   1/60  val_auc=0.6479  (best=0.6479)
  Epoch  10/60  val_auc=0.9250  (best=0.9708)
  Epoch  20/60  val_auc=0.9146  (best=0.9708)
  Epoch  30/60  val_auc=0.9333  (best=0.9708)
  Epoch  40/60  val_auc=0.9167  (best=0.9708)
  Epoch  50/60  val_auc=0.9250  (best=0.9708)
  Epoch  60/60  val_auc=0.9229  (best=0.9708)
  ✓ Fold 1: AUC=0.9708  Acc=0.8864

── Fold 2/5 ────────────────────────────────
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
  Epoch   1/60  val_auc=0.4667  (best=0.4667)
  Epoch  10/60  val_auc=0.9458  (best=0.9521)
  Epoch  20/60  val_auc=0.9188  (best=0.9521)
  Epoch  30/60  val_auc=0.9083  (best=0.9521)
  Epoch  40/60  val_auc=0.9083  (best=0.9521)
  Epoch  50/60  val_auc=0.9021  (best=0.9521)
  Epoch  60/60  val_auc=0.9021  (best=0.9521)
  ✓ Fold 2: AUC=0.9521  Acc=0.8864


Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
  Epoch   1/60  val_auc=0.6271  (best=0.6271)
  Epoch  10/60  val_auc=0.8792  (best=0.9146)
  Epoch  20/60  val_auc=0.9021  (best=0.9187)
  Epoch  30/60  val_auc=0.8979  (best=0.9396)
  Epoch  40/60  val_auc=0.9292  (best=0.9396)
  Epoch  50/60  val_auc=0.9208  (best=0.9417)
  Epoch  60/60  val_auc=0.9167  (best=0.9417)
  ✓ Fold 3: AUC=0.9417  Acc=0.8409

── Fold 4/5 ────────────────────────────────
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
  Epoch   1/60  val_auc=0.5104  (best=0.5104)
  Epoch  10/60  val_auc=0.8458  (best=0.9021)
  Epoch  20/60  val_auc=0.8771  (best=0.9021)
  Epoch  30/60  val_auc=0.8750  (best=0.9021)
  Epoch  40/60  val_auc=0.8500  (best=0.9021)
  Epoch  50/60  val_auc=0.8604  (best=0.9021)
  Epoch  60/60  val_auc=0.8583  (best=0.9021)
  ✓ Fold 4: AUC=0.9021  Acc=0.7955

── Fold 5/5 ────────────────────────────────
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
  Epoch   1/60  val_

In [24]:
import os

# ── Model weight directories ──────────────────────────────────
BASE_DIR = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis"

BARCHE_MODEL_DIR = os.path.join(
    BASE_DIR, "Tip_Features_Output", "barche_efficientnet"
)
TIPDOT_MODEL_DIR = os.path.join(
    BASE_DIR, "predicted_masks", "efficientnet_tip_dot_v3"
)

# ── Barche OOF results (from training log) ────────────────────
barche_oof_auc = roc_auc_score(labels_array, all_val_probs)
barche_oof_acc = accuracy_score(labels_array,
                                (all_val_probs >= 0.5).astype(int))

# ── Verify paths exist ────────────────────────────────────────
print("BARCHE_MODEL_DIR exists:", os.path.exists(BARCHE_MODEL_DIR))
print("TIPDOT_MODEL_DIR exists:", os.path.exists(TIPDOT_MODEL_DIR))

print(f"\nBarche OOF AUC : {barche_oof_auc:.4f}")
print(f"Barche OOF Acc : {barche_oof_acc:.4f}")

# Check fold weights are present
for fold in range(1, 6):
    b_path = os.path.join(BARCHE_MODEL_DIR, f"fold{fold}_best.pt")
    t_path = os.path.join(TIPDOT_MODEL_DIR, f"fold{fold}_best.pt")
    print(f"  Fold {fold} — Barche: {'✓' if os.path.exists(b_path) else '✗ MISSING'}  "
          f"Tip-dot: {'✓' if os.path.exists(t_path) else '✗ MISSING'}")

BARCHE_MODEL_DIR exists: True
TIPDOT_MODEL_DIR exists: True

Barche OOF AUC : 0.9296
Barche OOF Acc : 0.8591
  Fold 1 — Barche: ✓  Tip-dot: ✓
  Fold 2 — Barche: ✓  Tip-dot: ✓
  Fold 3 — Barche: ✓  Tip-dot: ✓
  Fold 4 — Barche: ✓  Tip-dot: ✓
  Fold 5 — Barche: ✓  Tip-dot: ✓


In [25]:
import pandas as pd

# --- Use the EXACT paths we just found ---
# 1. The CSV is directly inside 'predicted_masks'
ML_CSV = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/predicted_masks/catheter_tip_features_ml_ready.csv"

# 2. The Excel file is in the main 'Master Thesis' folder
LABEL_XLSX = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/atriumandcatheter.xlsx"

# --- Load data ---
print("Loading CSV...")
model_df = pd.read_csv(ML_CSV)

print("Loading Excel...")
labels_excel = pd.read_excel(LABEL_XLSX)[["ap_id", "tip (1-ok, 0-no)"]].copy()
labels_excel["ap_id"] = labels_excel["ap_id"].astype(str).str.strip()

# --- Extract clean ap_id ---
model_df["ap_id"] = (
    model_df["image_name"]
    .astype(str)
    .str.replace(r"\.png$", "", regex=True)
    .str.replace(r"\s*-\s*PRIMARY$", "", regex=True)
    .str.replace(r"\.dcm$", "", regex=True)
    .str.strip()
)

# --- Merge on ap_id ---
model_df = model_df.merge(labels_excel, on="ap_id", how="inner")
print(f"model_df        : {model_df.shape}")

Loading CSV...
Loading Excel...
model_df        : (220, 58)


In [26]:
print("model_df :", model_df.shape)
print("catheter_np :", catheter_np.shape)
print("atrium_np :", atrium_np.shape)
print("ids_list :", len(ids_list))

def make_tip_dot_image(original_shape, tip_coord, target_size, sigma=35):
    H_orig, W_orig = original_shape
    if tip_coord is None:
        return np.zeros((target_size, target_size), dtype=np.uint8)
    if np.isnan(float(tip_coord[0])) or np.isnan(float(tip_coord[1])):
        return np.zeros((target_size, target_size), dtype=np.uint8)
    ty_scaled    = float(tip_coord[0]) * target_size / H_orig
    tx_scaled    = float(tip_coord[1]) * target_size / W_orig
    sigma_scaled = sigma * target_size / H_orig
    y_grid, x_grid = np.mgrid[0:target_size,
                               0:target_size].astype(np.float32)
    canvas = np.exp(
        -((y_grid - ty_scaled)**2 + (x_grid - tx_scaled)**2)
        / (2.0 * sigma_scaled**2)
    )
    return (canvas * 255.0).astype(np.uint8)


def build_efficientnet_b7(dropout=0.6, device=None,
                           verbose=False):  # FIX: suppress print by default
    if device is None:
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
    model = timm.create_model(
        "tf_efficientnet_b7",
        pretrained=False,
        in_chans=3,
        num_classes=1,
    )
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if any(layer in name for layer in [
            "conv_head", "bn2", "classifier"
        ]):
            param.requires_grad = True
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1),
    )
    if verbose:  # FIX: only print when explicitly asked
        trainable = sum(p.numel() for p in model.parameters()
                        if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"EfficientNet-B7 [head_only]: "
              f"{trainable:,} / {total:,} trainable "
              f"({100*trainable/total:.1f}%)")
    return model.to(device)

# Quick test — prints once only
build_efficientnet_b7(verbose=True)
print("✓ Functions defined")

model_df : (220, 58)
catheter_np : (220, 960, 960)
atrium_np : (220, 960, 960)
ids_list : 220
EfficientNet-B7 [head_only]: 1,786,497 / 63,789,521 trainable (2.8%)
✓ Functions defined


In [27]:
from sklearn.metrics import precision_score, recall_score

def two_step_predict(model_df, catheter_np, atrium_np, ids_list,
                     barche_dir, tipdot_dir,
                     img_size=600,
                     sigma=35,
                     threshold=0.5,
                     device=None):
    """
    Routes each image to the correct model based on tip status.

    FIX: Separate transforms for each route
      Barche  → Resize only           (no normalize — Barche not trained with it)
      Tip-dot → Normalize(0.5, 0.5)   (v3 WAS trained with normalize)
    """
    if device is None:
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    id_to_idx = {name: i for i, name in enumerate(ids_list)}

    # ── FIX: TWO separate transforms ─────────────────────────────
    # Barche was trained WITHOUT normalization
    barche_transform = A.Compose([
        A.Resize(img_size, img_size,
                 interpolation=cv2.INTER_NEAREST),
        A.ToTensorV2()
    ])

    # v3 tip-dot was trained WITH normalization — this was missing before
    tipdot_transform = A.Compose([
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])
    # ─────────────────────────────────────────────────────────────

    all_probs  = []
    all_preds  = []
    all_routes = []

    total = len(model_df)
    for i, (_, row) in enumerate(model_df.iterrows()):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  Processing {i+1}/{total}...")

        img_name = row["image_name"]
        status   = row["status"]
        idx      = id_to_idx.get(img_name, None)

        if idx is None:
            all_probs.append(0.5)
            all_preds.append(0)
            all_routes.append("not_found")
            continue

        fold_probs = []

        for fold in range(1, 6):

            if status == "ambiguous_tip":
                # ── Route A: Barche full-mask ─────────────────────
                model = build_barche_efficientnet(
                    fine_tune="last_two", device=device
                )
                weight_path = os.path.join(
                    barche_dir, f"fold{fold}_best.pt"
                )
                route = "barche_full_mask"

                cath_raw    = catheter_np[idx].astype(np.float32)
                atrium_raw  = atrium_np[idx].astype(np.float32)
                blank       = np.zeros_like(cath_raw)
                feature_hwc = np.stack(
                    [cath_raw, atrium_raw, blank], axis=-1
                )
                # FIX: use barche_transform (resize, NO normalize)
                tensor = barche_transform(
                    image=feature_hwc
                )["image"].unsqueeze(0).to(device)

            else:
                # ── Route B: tip-dot v3 ───────────────────────────
                model = build_efficientnet_b7(
                    dropout=0.6, device=device,
                    verbose=False  # suppress the print
                )
                weight_path = os.path.join(
                    tipdot_dir, f"fold{fold}_best.pt"
                )
                route = "tipdot_v3"

                tip_coord  = (row["tip_y"], row["tip_x"])
                dot        = make_tip_dot_image(
                    (960, 960), tip_coord,
                    img_size, sigma
                ).astype(np.float32)
                atrium_res = cv2.resize(
                    atrium_np[idx].astype(np.float32),
                    (img_size, img_size),
                    interpolation=cv2.INTER_NEAREST
                )
                blank       = np.zeros_like(dot)
                feature_hwc = np.stack(
                    [dot, atrium_res, blank], axis=-1
                )
                # FIX: use tipdot_transform (WITH normalize)
                tensor = tipdot_transform(
                    image=feature_hwc
                )["image"].unsqueeze(0).to(device)

            # Load weights and predict
            model.load_state_dict(
                torch.load(weight_path, map_location=device)
            )
            model.eval()

            with torch.no_grad():
                prob = torch.sigmoid(
                    model(tensor).squeeze()
                ).item()

            fold_probs.append(prob)
            del model
            torch.cuda.empty_cache()

        avg_prob = float(np.mean(fold_probs))
        pred     = int(avg_prob >= threshold)

        all_probs.append(round(avg_prob, 4))
        all_preds.append(pred)
        all_routes.append(route)

    return np.array(all_probs), np.array(all_preds), all_routes

In [28]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score,
    roc_auc_score, accuracy_score, f1_score
)
import numpy as np

print("Running two-step classifier...")
print("Loads 5 fold models per image — takes ~10 minutes\n")

y_true = model_df["tip (1-ok, 0-no)"].astype(int).values

two_step_probs, two_step_preds, routes = two_step_predict(
    model_df     = model_df,
    catheter_np  = catheter_np,
    atrium_np    = atrium_np,
    ids_list     = ids_list,
    barche_dir   = BARCHE_MODEL_DIR,
    tipdot_dir   = TIPDOT_MODEL_DIR,
    img_size     = 600,
    sigma        = 35,
    threshold    = 0.5,
    device       = device,
)

# ── Core metrics ──────────────────────────────────────────────
ts_auc  = roc_auc_score(y_true, two_step_probs)
ts_acc  = accuracy_score(y_true, two_step_preds)
ts_prec = precision_score(y_true, two_step_preds, zero_division=0)
ts_rec  = recall_score(y_true, two_step_preds, zero_division=0)
ts_f1   = f1_score(y_true, two_step_preds, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_true, two_step_preds).ravel()
ts_spec = tn / (tn + fp)

routes_list = list(routes)
n_barche = routes_list.count("barche_full_mask")
n_tipdot = routes_list.count("tipdot_v3")
n_missing = routes_list.count("not_found")

print("=" * 65)
print("TWO-STEP CLASSIFIER — RESULTS")
print("=" * 65)
print(f"  Routed to Barche full-mask : {n_barche}  (ambiguous_tip)")
print(f"  Routed to tip-dot v3       : {n_tipdot}  (success + overlap)")
print(f"  Not found in mask arrays   : {n_missing}")
print(f"\n  AUC         : {ts_auc:.4f}")
print(f"  Accuracy    : {ts_acc:.4f}")
print(f"  Precision   : {ts_prec:.4f}")
print(f"  Sensitivity : {ts_rec:.4f}")
print(f"  Specificity : {ts_spec:.4f}")
print(f"  F1          : {ts_f1:.4f}")

print("\n" + classification_report(
    y_true, two_step_preds,
    target_names=["Malpositioned (0)", "Correct (1)"]
))

# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_true, two_step_preds)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Malpositioned (0)", "Correct (1)"]
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(
    f"Two-Step Classifier\n"
    f"AUC={ts_auc:.3f}  Acc={ts_acc:.3f}  "
    f"Sens={ts_rec:.3f}  Spec={ts_spec:.3f}",
    fontsize=11
)
plt.tight_layout()
plt.savefig("two_step_confusion.png", dpi=150)
plt.show()

# ── Ambiguous tip class distribution ─────────────────────────
print("=== Ambiguous tip — class distribution ===")
amb = model_df[model_df["status"] == "ambiguous_tip"]["tip (1-ok, 0-no)"]
print(amb.value_counts().rename({0: "Malpositioned (0)", 1: "Correct (1)"}))
print(f"Total ambiguous: {len(amb)}")


# ── Two-step error analysis by status ────────────────────────
print("\n=== Two-step error analysis by status ===")
result_df = model_df[["image_name", "status", "tip (1-ok, 0-no)"]].copy()
result_df["prob"]       = two_step_probs
result_df["pred"]       = two_step_preds
result_df["model_used"] = routes_list

for s in ["success", "success_overlap_mode", "ambiguous_tip"]:
    sub = result_df[result_df["status"] == s]
    if len(sub) == 0:
        continue
    sub_auc = (
        roc_auc_score(sub["tip (1-ok, 0-no)"], sub["prob"])
        if len(sub["tip (1-ok, 0-no)"].unique()) > 1
        else float("nan")
    )
    sub_acc  = accuracy_score(sub["tip (1-ok, 0-no)"], sub["pred"])
    sub_f1   = f1_score(sub["tip (1-ok, 0-no)"], sub["pred"], zero_division=0)
    tn_s, fp_s, fn_s, tp_s = confusion_matrix(
        sub["tip (1-ok, 0-no)"], sub["pred"]
    ).ravel() if len(sub["tip (1-ok, 0-no)"].unique()) > 1 else (0, 0, 0, len(sub))
    sub_spec = tn_s / (tn_s + fp_s) if (tn_s + fp_s) > 0 else float("nan")
    print(f"  {s:<25}  n={len(sub):3d}  "
          f"Acc={sub_acc:.3f}  AUC={sub_auc:.3f}  "
          f"Sens={sub_f1:.3f}  Spec={sub_spec:.3f}")

save_path = os.path.join(BARCHE_MODEL_DIR, "two_step_results.csv")
result_df.to_csv(save_path, index=False)
print(f"\nSaved: {save_path}")

Running two-step classifier...
Loads 5 fold models per image — takes ~10 minutes

  Processing 1/220...
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51,524,473 / 63,789,521 trainable
Barche EfficientNet-B7: 51

NameError: name 'lr_oof_auc' is not defined

In [29]:

# ── Two-step error analysis by status ────────────────────────
print("\n=== Two-step error analysis by status ===")
result_df = model_df[["image_name", "status", "tip (1-ok, 0-no)"]].copy()
result_df["prob"]       = two_step_probs
result_df["pred"]       = two_step_preds
result_df["model_used"] = routes_list

for s in ["success", "success_overlap_mode", "ambiguous_tip"]:
    sub = result_df[result_df["status"] == s]
    if len(sub) == 0:
        continue
    sub_auc = (
        roc_auc_score(sub["tip (1-ok, 0-no)"], sub["prob"])
        if len(sub["tip (1-ok, 0-no)"].unique()) > 1
        else float("nan")
    )
    sub_acc  = accuracy_score(sub["tip (1-ok, 0-no)"], sub["pred"])
    sub_f1   = f1_score(sub["tip (1-ok, 0-no)"], sub["pred"], zero_division=0)
    tn_s, fp_s, fn_s, tp_s = confusion_matrix(
        sub["tip (1-ok, 0-no)"], sub["pred"]
    ).ravel() if len(sub["tip (1-ok, 0-no)"].unique()) > 1 else (0, 0, 0, len(sub))
    sub_spec = tn_s / (tn_s + fp_s) if (tn_s + fp_s) > 0 else float("nan")
    print(f"  {s:<25}  n={len(sub):3d}  "
          f"Acc={sub_acc:.3f}  AUC={sub_auc:.3f}  "
          f"Sens={sub_f1:.3f}  Spec={sub_spec:.3f}")

save_path = os.path.join(BARCHE_MODEL_DIR, "two_step_results.csv")
result_df.to_csv(save_path, index=False)
print(f"\nSaved: {save_path}")


=== Two-step error analysis by status ===
  success                    n=197  Acc=0.909  AUC=0.963  Sens=0.899  Spec=0.892
  success_overlap_mode       n=  3  Acc=0.667  AUC=0.500  Sens=0.667  Spec=1.000
  ambiguous_tip              n= 20  Acc=1.000  AUC=1.000  Sens=1.000  Spec=1.000

Saved: /vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/Tip_Features_Output/barche_efficientnet/two_step_results.csv


In [30]:
print("=== Ambiguous tip — class distribution ===")
print(model_df[model_df["status"] == "ambiguous_tip"]["tip (1-ok, 0-no)"].value_counts())
print(f"Total: {len(model_df[model_df['status'] == 'ambiguous_tip'])}")

=== Ambiguous tip — class distribution ===
tip (1-ok, 0-no)
1    13
0     7
Name: count, dtype: int64
Total: 20


In [7]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score
from sklearn.model_selection import StratifiedKFold

BASE_DIR         = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis"
SAVE_DIR         = os.path.join(BASE_DIR, "Tip_Features_Output", "barche_efficientnet")
BARCHE_MODEL_DIR = SAVE_DIR
TIPDOT_DIR       = os.path.join(BASE_DIR, "predicted_masks", "efficientnet_tip_dot_v3")

print("Barche dir exists:", os.path.exists(SAVE_DIR))
print("Tip-dot dir exists:", os.path.exists(TIPDOT_DIR))
print("\nBarche files:", os.listdir(SAVE_DIR))
print("Tip-dot files:", os.listdir(TIPDOT_DIR))

Barche dir exists: True
Tip-dot dir exists: True

Barche files: ['fold1_best.pt', 'fold2_best.pt', 'fold3_best.pt', 'fold4_best.pt', 'fold5_best.pt', 'oof_probs.npy', 'oof_preds.npy', 'two_step_results.csv']
Tip-dot files: ['fold1_best.pt', 'fold2_best.pt', 'fold3_best.pt', 'fold4_best.pt', 'fold5_best.pt', 'oof_predictions_v3.csv']


In [10]:
import subprocess
subprocess.run(["pip", "install", "openpyxl",
                "--target", "/data/leuven/375/vsc37504/thesis_libs",
                "--quiet"])
print("Done — restart kernel and rerun")

Done — restart kernel and rerun



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
ML_CSV     = os.path.join(BASE_DIR, "predicted_masks",
                          "catheter_tip_features_ml_ready.csv")
LABEL_XLSX = os.path.join(BASE_DIR, "atriumandcatheter.xlsx")

model_df     = pd.read_csv(ML_CSV)
labels_excel = pd.read_excel(LABEL_XLSX)[["ap_id", "tip (1-ok, 0-no)"]].copy()
labels_excel["ap_id"] = labels_excel["ap_id"].astype(str).str.strip()

model_df["ap_id"] = (
    model_df["image_name"].astype(str)
    .str.replace(r"\.png$", "", regex=True)
    .str.replace(r"\s*-\s*PRIMARY$", "", regex=True)
    .str.replace(r"\.dcm$", "", regex=True)
    .str.strip()
)
model_df     = model_df.merge(labels_excel, on="ap_id", how="inner")
labels_array = model_df["tip (1-ok, 0-no)"].astype(int).values

print(f"labels_array ready: {len(labels_array)} samples")
print(f"Class balance — 0: {(labels_array==0).sum()},  1: {(labels_array==1).sum()}")

labels_array ready: 220 samples
Class balance — 0: 119,  1: 101


In [9]:
barche_probs = np.load(os.path.join(SAVE_DIR, "oof_probs.npy"))
barche_preds = np.load(os.path.join(SAVE_DIR, "oof_preds.npy"))

print(f"Barche OOF AUC = {roc_auc_score(labels_array, barche_probs):.4f}")
print(f"Barche OOF Acc = {accuracy_score(labels_array, barche_preds):.4f}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sens_folds, spec_folds = [], []

for _, val_idx in skf.split(np.arange(len(labels_array)), labels_array):
    yv = labels_array[val_idx]
    yp = barche_preds[val_idx].astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, yp).ravel()
    sens_folds.append(tp / (tp + fn))
    spec_folds.append(tn / (tn + fp))

print(f"\nBarche Sensitivity : {np.mean(sens_folds):.3f}  "
      f"SE = {np.std(sens_folds)/np.sqrt(5):.3f}")
print(f"Barche Specificity : {np.mean(spec_folds):.3f}  "
      f"SE = {np.std(spec_folds)/np.sqrt(5):.3f}")

Barche OOF AUC = 0.9296
Barche OOF Acc = 0.8591

Barche Sensitivity : 0.890  SE = 0.046
Barche Specificity : 0.832  SE = 0.037


In [13]:
ts_csv    = os.path.join(BARCHE_MODEL_DIR, "two_step_results.csv")
result_df = pd.read_csv(ts_csv)

y_true = result_df["tip (1-ok, 0-no)"].astype(int).values
y_pred = result_df["pred"].astype(int).values
y_prob = result_df["prob"].values

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
ts_sens = tp / (tp + fn)
ts_spec = tn / (tn + fp)
ts_auc  = roc_auc_score(y_true, y_prob)
ts_acc  = accuracy_score(y_true, y_pred)

print(f"Two-step AUC         : {ts_auc:.4f}")
print(f"Two-step Accuracy    : {ts_acc:.4f}")
print(f"Two-step Sensitivity : {ts_sens:.3f}")
print(f"Two-step Specificity : {ts_spec:.3f}")
print("\n(single OOF pass — no per-fold SE available for two-step)")

Two-step AUC         : 0.9645
Two-step Accuracy    : 0.9136
Two-step Sensitivity : 0.931
Two-step Specificity : 0.899

(single OOF pass — no per-fold SE available for two-step)


In [10]:
print("=" * 55)
print("SUMMARY — ALL MODELS SENSITIVITY / SPECIFICITY")
print("=" * 55)
print(f"\nBarche full-mask:")
print(f"  Sensitivity : {np.mean(sens_folds):.3f}  SE={np.std(sens_folds)/np.sqrt(5):.3f}")
print(f"  Specificity : {np.mean(spec_folds):.3f}  SE={np.std(spec_folds)/np.sqrt(5):.3f}")
print(f"\nTwo-step classifier:")
print(f"  Sensitivity : {ts_sens:.3f}")
print(f"  Specificity : {ts_spec:.3f}")

SUMMARY — ALL MODELS SENSITIVITY / SPECIFICITY

Barche full-mask:
  Sensitivity : 0.890  SE=0.046
  Specificity : 0.832  SE=0.037

Two-step classifier:


NameError: name 'ts_sens' is not defined

In [11]:
import pandas as pd

# ── Load saved two-step predictions ──────────────────────────
ts_csv    = os.path.join(BARCHE_MODEL_DIR, "two_step_results.csv")
result_df = pd.read_csv(ts_csv)

y_true          = result_df["tip (1-ok, 0-no)"].astype(int).values
two_step_preds  = result_df["pred"].astype(int).values

# ── Per-fold SE ───────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ts_sens_folds, ts_spec_folds = [], []

for _, val_idx in skf.split(np.arange(len(labels_array)), labels_array):
    yv = labels_array[val_idx]
    yp = two_step_preds[val_idx]
    tn, fp, fn, tp = confusion_matrix(yv, yp).ravel()
    ts_sens_folds.append(tp / (tp + fn))
    ts_spec_folds.append(tn / (tn + fp))

print(f"Two-step Sensitivity : {np.mean(ts_sens_folds):.3f}  "
      f"SE = {np.std(ts_sens_folds)/np.sqrt(5):.3f}")
print(f"Two-step Specificity : {np.mean(ts_spec_folds):.3f}  "
      f"SE = {np.std(ts_spec_folds)/np.sqrt(5):.3f}")

Two-step Sensitivity : 0.931  SE = 0.033
Two-step Specificity : 0.899  SE = 0.015


In [13]:
import pandas as pd
import os

BASE_DIR = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis"
ML_CSV = os.path.join(BASE_DIR, "predicted_masks", "catheter_tip_features_ml_ready.csv")

df = pd.read_csv(ML_CSV)
print("Columns in CSV:")
print(df.columns.tolist())
print(f"\nRows: {len(df)}")

Columns in CSV:
['image_name', 'status', 'tip_y', 'tip_x', 'tip_pixels', 'atrium_pixels', 'overlap_pixels', 'tip_area_inside_ratio', 'atrium_coverage_ratio', 'iou_tip_atrium', 'tip_inside_atrium', 'centerline_inside_ratio', 'tip_dx', 'tip_dy', 'tip_angle', 'bbox_dy_top', 'bbox_dy_bottom', 'bbox_dx_left', 'bbox_dx_right', 'bbox_dy_top_norm', 'bbox_dy_bottom_norm', 'bbox_dx_left_norm', 'bbox_dx_right_norm', 'bbox_vertical_inside', 'bbox_horizontal_inside', 'bbox_inside_atrium', 'edge_dy_top_min', 'edge_dy_top_mean', 'edge_dy_bottom_min', 'edge_dy_bottom_mean', 'edge_dx_left_min', 'edge_dx_left_mean', 'edge_dx_right_min', 'edge_dx_right_mean', 'edge_dy_top_min_norm', 'edge_dy_top_mean_norm', 'edge_dy_bottom_min_norm', 'edge_dy_bottom_mean_norm', 'edge_dx_left_min_norm', 'edge_dx_left_mean_norm', 'edge_dx_right_min_norm', 'edge_dx_right_mean_norm', 'region_nearest_euclidean', 'region_nearest_dy', 'region_nearest_dx', 'region_nearest_dist_norm_h', 'region_nearest_dist_norm_w', 'mean_tip_rad

In [15]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, recall_score, confusion_matrix

BASE_DIR = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis"
SAVE_DIR = os.path.join(BASE_DIR, "Tip_Features_Output", "barche_efficientnet")

barche_probs = np.load(os.path.join(SAVE_DIR, "oof_probs.npy"))
barche_preds = np.load(os.path.join(SAVE_DIR, "oof_preds.npy")).astype(int)

# Load features
ML_CSV = os.path.join(BASE_DIR, "predicted_masks", "catheter_tip_features_ml_ready.csv")
model_df = pd.read_csv(ML_CSV)

# Load labels from Excel
LABEL_XLSX = os.path.join(BASE_DIR, "atriumandcatheter.xlsx")
labels_excel = pd.read_excel(LABEL_XLSX)[["ap_id", "tip (1-ok, 0-no)"]].copy()
labels_excel["ap_id"] = labels_excel["ap_id"].astype(str).str.strip()

# Build ap_id in model_df to merge on
model_df["ap_id"] = (
    model_df["image_name"].astype(str)
    .str.replace(r"\.png$", "", regex=True)
    .str.replace(r"\s*-\s*PRIMARY$", "", regex=True)
    .str.replace(r"\.dcm$", "", regex=True)
    .str.strip()
)
model_df = model_df.merge(labels_excel, on="ap_id", how="inner")

y = model_df["tip (1-ok, 0-no)"].astype(int).values

print(f"Samples: {len(y)}  (should be 220)")
print(f"F1          : {f1_score(y, barche_preds, zero_division=0):.4f}")
print(f"AUC         : {roc_auc_score(y, barche_probs):.4f}")
print(f"Accuracy    : {accuracy_score(y, barche_preds):.4f}")
print(f"Sensitivity : {recall_score(y, barche_preds, zero_division=0):.4f}")
tn, fp, fn, tp = confusion_matrix(y, barche_preds).ravel()
print(f"Specificity : {tn/(tn+fp):.4f}")

Samples: 220  (should be 220)
F1          : 0.8531
AUC         : 0.9296
Accuracy    : 0.8591
Sensitivity : 0.8911
Specificity : 0.8319
